# Human Review Comparison


## 1) Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2) Install Dependencies


In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn


## 3) Load Dataset


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = '/content/drive/MyDrive/Deep Learning Project/AI Agentic/data/processed'
required = ['X_train.npy','X_test.npy','y_train.npy','y_test.npy']
print('Data path:', DATA_PATH)
print('Files:', os.listdir(DATA_PATH))
missing = [f for f in required if not os.path.exists(os.path.join(DATA_PATH,f))]
if missing:
    raise FileNotFoundError(f'Missing required files: {missing}')

X_test=np.load(os.path.join(DATA_PATH,'X_test.npy'))
y_test=np.load(os.path.join(DATA_PATH,'y_test.npy')).reshape(-1)
if X_test.ndim==3 and X_test.shape[-1]==1: X_test=X_test[...,0]



## 4) Generate Attack Predictions


In [ ]:
def gen_pred(X,y,n=500,seed=42):
    rng=np.random.default_rng(seed)
    idx=rng.choice(len(y),size=min(n,len(y)),replace=False)
    rows=[]
    for sid,i in enumerate(idx):
        yi=int(y[i]); conf=float(np.clip(0.65+0.2*(yi>0)+rng.normal(0,0.15),0,1))
        risk=float(np.clip(0.6*conf+0.3*(yi>0)+rng.normal(0,0.1),0,1))
        gs=float(np.clip(rng.beta(2,5) if yi>0 else rng.beta(5,2),0,1))
        dec='BLOCK_IP' if risk>0.8 else 'RATE_LIMIT' if risk>0.6 else 'ALERT_ADMIN' if risk>0.4 else 'ALLOW'
        rows.append({'sample_id':sid,'true_label':yi,'confidence':conf,'risk_score':risk,'graph_similarity':gs,'decision':dec})
    return pd.DataFrame(rows)

df=gen_pred(X_test,y_test)
df.head()



## 5) Triggers + Human Input


In [ ]:
def t1(r): return r['confidence']<0.65
def t2(r): return 0.45<=r['risk_score']<=0.65
def t3(r): return r['graph_similarity']<0.35
strategies={'Confidence Threshold Trigger':t1,'Risk Uncertainty Trigger':t2,'Novel Attack Trigger (Graph Similarity)':t3}

ans=input('Approve system decision? yes/no: ').strip().lower()
if ans not in ['yes','no']:
    ans='yes'

store={}
for k,fn in strategies.items():
    t=df.copy()
    t['review_required']=t.apply(fn,axis=1)
    t['human_decision']=np.where(t['review_required'],ans,'auto_accept')
    store[k]=t
print('Stored human decisions for',len(store),'strategies')



## 6) Compare Review Frequency and FP Reduction


In [ ]:
rows=[]
for k,t in store.items():
    rf=float(t['review_required'].mean())
    ben=t[t['true_label']==0]
    baseline=((ben['decision']!='ALLOW')).sum()
    corrected=((ben['review_required']) & (ben['human_decision']=='no') & (ben['decision']!='ALLOW')).sum()
    fpr=float(corrected/max(int(baseline),1)) if len(ben)>0 else 0.0
    rows.append({'Method':k,'ReviewFrequency':rf,'FalsePositiveReduction':fpr})
res=pd.DataFrame(rows).sort_values(['FalsePositiveReduction','ReviewFrequency'],ascending=[False,True]).reset_index(drop=True)
res



## 7) Charts


In [ ]:
res.plot(x='Method',y='ReviewFrequency',kind='bar',figsize=(10,4),ylim=(0,1),title='Review Frequency')
plt.grid(axis='y',alpha=0.3); plt.tight_layout(); plt.show()
res.plot(x='Method',y='FalsePositiveReduction',kind='bar',figsize=(10,4),ylim=(0,1),title='False Positive Reduction')
plt.grid(axis='y',alpha=0.3); plt.tight_layout(); plt.show()

